# Small Groups of Order 1–20, Built with `finalg`

This notebook builds a dictionary, `small_groups`, whose keys are the orders 1 through 20 and whose
values are lists of all groups of that order, one per isomorphism class (54 groups in all). It follows
the reference table in `small_groups_table.ipynb`.

Wherever possible, groups are built from simpler ones using the direct product (`*`, `**`). The
remaining groups come from `finalg`'s generators, from `Group.semidirect_product` ($\rtimes$), and from
`Group.central_product` ($\circ$), which is needed only for the Pauli group $Z_4\circ D_4$ of order 16.

In [1]:
import itertools as it
from collections import Counter

import finalg as fg
from finalg.group import Group

Finite Algebras:
  To see a listing of built-in example algebras, run finalg.examples.about()
  To retrieve an example algebra, use finalg.examples[INDEX]



## Helpers

In [2]:
def z(n):
    """Return the cyclic group of order n, where n is a positive integer."""
    return fg.generate_cyclic_group(n)


def mult_action(modulus, k):
    """Action of Z_m (string-named integer elements) on Z_modulus, where h acts on x by x -> k^h * x (mod modulus)."""
    return lambda h, x: str((int(x) * pow(k, int(h), modulus)) % modulus)


def central_involution(grp):
    """Return the one non-identity central element of a group whose center has order 2."""
    ctr = [c for c in grp.center() if c != grp.identity]
    assert len(ctr) == 1, f"{grp.name} does not have a center of order 2"
    return ctr[0]

## Groups from `finalg` generators and direct products

In [3]:
v4 = z(2) * z(2); v4.name = 'V4'; v4.description += ' (Klein-4 group)'
d3, _, _ = fg.generate_dihedral_group(3)    # d3 \equiv s3 (order 6)
d4, _, _ = fg.generate_dihedral_group(4)    # order 8
d5, _, _ = fg.generate_dihedral_group(5)    # order 10
d6, _, _ = fg.generate_dihedral_group(6)    # order 12
d7, _, _ = fg.generate_dihedral_group(7)    # order 14
d8, _, _ = fg.generate_dihedral_group(8)    # order 16
d9, _, _ = fg.generate_dihedral_group(9)    # order 18
d10, _, _ = fg.generate_dihedral_group(10)  # order 20
a4 = fg.generate_alternating_group(4)
q8, _, _ = fg.generate_dicyclic_group(2)    # Dic2 = Q8
dic3, _, _ = fg.generate_dicyclic_group(3)  # \equiv Z3 X| Z4, Z4 acting by inversion
dic4, _, _ = fg.generate_dicyclic_group(4)  # Dic4 = Q16
dic5, _, _ = fg.generate_dicyclic_group(5)  # \equiv Z5 X| Z4, Z4 acting by inversion

## Groups that need semidirect or central products

**Order 16**

- $Z_4\rtimes Z_4$: the generator of the second factor inverts the first.
- $(Z_2\times Z_2)\rtimes Z_4$: the generator of $Z_4$ swaps the two $Z_2$ factors.
- $M_{16}=Z_8\rtimes Z_2$ with $x\mapsto x^5$, and $SD_{16}=Z_8\rtimes Z_2$ with $x\mapsto x^3$.
- The Pauli group $Z_4\circ D_4$ ($\cong Z_4\circ Q_8$), identifying the order-2 element of $Z_4$ with the central involution of $D_4$.

**Order 18:** the generalized dihedral group $(Z_3\times Z_3)\rtimes Z_2$, where $Z_2$ inverts every element.

**Order 20:** the Frobenius group $F_{20}=Z_5\rtimes Z_4$, where the generator of $Z_4$ acts by $x\mapsto x^2$ (a faithful action).

In [4]:
# Order 16
z4_sdp_z4 = z(4).semidirect_product(z(4), mult_action(4, 3), name='Z4_sdp_Z4')           # Z4 X| Z4, inversion
m16  = z(8).semidirect_product(z(2), mult_action(8, 5), name='M16')                       # Z8 X| Z2, x -> x^5
sd16 = z(8).semidirect_product(z(2), mult_action(8, 3), name='SD16')                      # Z8 X| Z2, x -> x^3

swap = lambda h, x: x if int(h) % 2 == 0 else ':'.join(reversed(x.split(':')))
z2z2_sdp_z4 = (z(2) * z(2)).semidirect_product(z(4), swap, name='Z2Z2_sdp_Z4')            # (Z2 x Z2) X| Z4, swap

pauli = z(4).central_product(d4, ('2', central_involution(d4)), name='Pauli')            # Z4 o D4 (= Z4 o Q8)

# Order 18
invert = lambda h, x: x if int(h) % 2 == 0 else ':'.join(str((-int(t)) % 3) for t in x.split(':'))
dih_z3z3 = (z(3) * z(3)).semidirect_product(z(2), invert, name='Dih_Z3xZ3')              # (Z3 x Z3) X| Z2, inversion

# Order 20
f20 = z(5).semidirect_product(z(4), mult_action(5, 2), name='F20')                        # Z5 X| Z4, faithful action

## The dictionary

In [5]:
small_groups = {
    1: [z(1)],
    2: [z(2)],
    3: [z(3)],
    4: [z(4), v4],
    5: [z(5)],
    6: [z(6), d3],  # d3 \equiv s3
    7: [z(7)],
    8: [z(8), z(4)*z(2), z(2)**3, d4, q8],
    9: [z(9), z(3)*z(3)],
    10: [z(10), d5],
    11: [z(11)],
    12: [z(12), z(6)*z(2), a4, d6, dic3],  # dic3 \equiv Z3 X| Z4
    13: [z(13)],
    14: [z(14), d7],
    15: [z(15)],
    16: [z(16), z(4)*z(4), z2z2_sdp_z4, z4_sdp_z4, z(8)*z(2), m16, d8, sd16, dic4,
         z(4)*z(2)**2, d4*z(2), q8*z(2), pauli, z(2)**4],
    17: [z(17)],
    18: [z(18), d9, d3*z(3), dih_z3z3, z(6)*z(3)],
    19: [z(19)],
    20: [z(20), z(10)*z(2), d10, dic5, f20],  # dic5 \equiv Z5 X| Z4 (inversion), f20 = Z5 X| Z4 (faithful)
}

## Checks

For each order, the number of groups must match the known count (1, 1, 1, 2, 1, 2, 1, 5, 2, 2, 1, 5, 1, 2, 1, 14, 1, 5, 1, 5),
every group must have that order, the number of abelian groups must match, and no two groups of the same order may be
isomorphic (tested with `fast_isomorphic`). Together, these show that each list has exactly one group from each isomorphism class.

In [6]:
EXPECTED_COUNTS = [1, 1, 1, 2, 1, 2, 1, 5, 2, 2, 1, 5, 1, 2, 1, 14, 1, 5, 1, 5]
EXPECTED_ABELIAN = [1, 1, 1, 2, 1, 1, 1, 3, 2, 1, 1, 2, 1, 1, 1, 5, 1, 2, 1, 2]

assert sorted(small_groups) == list(range(1, 21))
for n, grps in small_groups.items():
    assert len(grps) == EXPECTED_COUNTS[n - 1], f"order {n}: {len(grps)} groups"
    assert all(isinstance(g, Group) and g.order == n for g in grps), f"order {n}: wrong type or order"
    assert sum(g.is_commutative() for g in grps) == EXPECTED_ABELIAN[n - 1], f"order {n}: abelian count"
    for g1, g2 in it.combinations(grps, 2):
        assert not g1.fast_isomorphic(g2), f"order {n}: {g1.name} and {g2.name} are isomorphic"

total = sum(len(v) for v in small_groups.values())
assert total == 54
print(f"All checks passed: {total} groups of order 1-20, no two isomorphic within an order.")

All checks passed: 54 groups of order 1-20, no two isomorphic within an order.


## Summary

Element-order profiles, as `(order, how many elements)`, are an easy cross-check against the reference table.

In [7]:
for n, grps in small_groups.items():
    for g in grps:
        profile = sorted(Counter(g.element_order(e) for e in g).items())
        kind = 'abelian    ' if g.is_commutative() else 'non-abelian'
        print(f"{n:>2}  {g.name:<20} {kind}  {profile}")

 1  Z1                   abelian      [(1, 1)]
 2  Z2                   abelian      [(1, 1), (2, 1)]
 3  Z3                   abelian      [(1, 1), (3, 2)]
 4  Z4                   abelian      [(1, 1), (2, 1), (4, 2)]
 4  V4                   abelian      [(1, 1), (2, 3)]
 5  Z5                   abelian      [(1, 1), (5, 4)]
 6  Z6                   abelian      [(1, 1), (2, 1), (3, 2), (6, 2)]
 6  D3                   non-abelian  [(1, 1), (2, 3), (3, 2)]
 7  Z7                   abelian      [(1, 1), (7, 6)]
 8  Z8                   abelian      [(1, 1), (2, 1), (4, 2), (8, 4)]
 8  Z4_x_Z2              abelian      [(1, 1), (2, 3), (4, 4)]
 8  Z2_x_Z2_x_Z2         abelian      [(1, 1), (2, 7)]
 8  D4                   non-abelian  [(1, 1), (2, 5), (4, 2)]
 8  Q8                   non-abelian  [(1, 1), (2, 1), (4, 6)]
 9  Z9                   abelian      [(1, 1), (3, 2), (9, 6)]
 9  Z3_x_Z3              abelian      [(1, 1), (3, 8)]
10  Z10                  abelian      [(1, 1), (